In [1]:
import pandas as pd
import numpy as np
from catnip.fla_redshift import FLA_Redshift
from sqlalchemy import null
from datetime import datetime

from prefect.blocks.system import Secret
from typing import Dict
from concurrent.futures import ThreadPoolExecutor

In [2]:
def get_redshift_credentials() -> Dict:

    cred_dict = {
        "dbname": Secret.load("stellar-redshift-db-name").get(),
        "host": Secret.load("stellar-redshift-host").get(),
        "port": 5439,
        "user": Secret.load("stellar-redshift-user-name").get(),
        "password": Secret.load("stellar-redshift-password").get(),

        "aws_access_key_id": Secret.load("fla-s3-aws-access-key-id-east-1").get(),
        "aws_secret_access_key": Secret.load("fla-s3-aws-secret-access-key-east-1").get(),
        "bucket": Secret.load("fla-s3-bucket-name-east-1").get(),
        "subdirectory": "us-east-1",

        "verbose": False,
    }

    return cred_dict

with ThreadPoolExecutor(1) as pool:
    rs_creds = pool.submit(lambda: get_redshift_credentials()).result()

In [ ]:
# what is the cutoff date year to year?
# do we want to group programs by larger type or location?
# what is the goal of managing the data?

# try hockey for free has no times (first tab)
# school raffles and PE visits blank

In [ ]:
# 1. combine all programs
# 2. just have email and join date
# 3. have function for calculating season
# 4.  

In [ ]:
# df = pd.read_csv("C:\\Users\\riffere\\Desktop\\community_programs.csv")
# FLA_Redshift(**rs_creds).write_to_warehouse(df = df, table_name= "community_full_data")

In [3]:
q = """
    WITH initial AS (
        SELECT
            LOWER(email) AS email,
            program_group,
            program,
            MIN(transaction_date::date) AS transaction_date,
              CASE
                WHEN date_part('month', MIN(transaction_date::date)) < 9 THEN
                  (date_part('year', MIN(transaction_date::date)) - 1)::int
                  || '-' ||
                  RIGHT(date_part('year', MIN(transaction_date::date))::int,2)
                ELSE
                  date_part('year', MIN(transaction_date::date))::int
                  || '-' ||
                  RIGHT((date_part('year', MIN(transaction_date::date)) + 1)::int,2)
              END AS season
        FROM
            custom.community_full_data
        GROUP BY
            email,
            program_group,
            program
    ),
    acct_rev_info AS (
        SELECT
            LOWER(c.email) AS email,
            t.season,
            c.createdon,
            SUM(t.gross_revenue) AS gross_revenue
        FROM
            custom.cth_v_historical_ticket t
        LEFT JOIN
            custom.korepss_externalsystemtocontact ext
            ON t.purchaser_ticketing_id = ext.externalcontactid
        LEFT JOIN
            custom.korepss_contacts c
            ON ext.crmcontactid = c.contactid
        GROUP BY
            c.email,
            t.season,
            c.createdon
    ),
    subs AS (
        SELECT
            LOWER(c.email) AS email,
            '2023-24' AS season,
            1 AS is_stm
        FROM
            custom.cth_v_ticket_subscription_2324 subs
        LEFT JOIN
            custom.korepss_externalsystemtocontact ext
            ON subs.purch_client_crm_id = ext.externalcontactid
        LEFT JOIN
            custom.korepss_contacts c
            ON ext.crmcontactid = c.contactid
        GROUP BY
            c.email
        UNION ALL
        SELECT
            LOWER(c.email) AS email,
            '2024-25' AS season,
            1 AS is_stm
        FROM
            custom.cth_v_ticket_subscription_2425 subs
        LEFT JOIN
            custom.korepss_externalsystemtocontact ext
            ON subs.purch_client_crm_id = ext.externalcontactid
        LEFT JOIN
            custom.korepss_contacts c
            ON ext.crmcontactid = c.contactid
        GROUP BY
            c.email
        UNION ALL
        SELECT
            LOWER(c.email) AS email,
            '2025-26' AS season,
            1 AS is_stm
        FROM
            custom.cth_v_ticket_subscription_2526 subs
        LEFT JOIN
            custom.korepss_externalsystemtocontact ext
            ON subs.purch_client_crm_id = ext.externalcontactid
        LEFT JOIN
            custom.korepss_contacts c
            ON ext.crmcontactid = c.contactid
        GROUP BY
            c.email
        UNION ALL
        SELECT
            LOWER(c.email) AS email,
            '2026-27' AS season,
            1 AS is_stm
        FROM
            custom.cth_v_ticket_subscription_2627 subs
        LEFT JOIN
            custom.korepss_externalsystemtocontact ext
            ON subs.purch_client_crm_id = ext.externalcontactid
        LEFT JOIN
            custom.korepss_contacts c
            ON ext.crmcontactid = c.contactid
        GROUP BY
            c.email
    )
    SELECT
        i.email,
        i.program_group,
        i.program,
        i.season AS first_program_season,
        a.season AS rev_stm_season,
        CASE
            WHEN createdon >= transaction_date THEN 'new_email'
            ELSE 'email_seen_before'
        END AS email_status,
        CASE
            WHEN first_program_season > rev_stm_season THEN 'pre_program_data'
            WHEN first_program_season = rev_stm_season THEN 'during_program_data'
            ELSE 'post_program_data'
        END AS program_data,
        COALESCE(a.gross_revenue,0) AS gross_rev,
        CASE
            WHEN ROW_NUMBER() OVER (
                PARTITION BY a.season, i.email
                ORDER BY i.program
            ) = 1
            THEN a.gross_revenue
            ELSE 0
        END AS gross_rev_dedup,
        COALESCE(s.is_stm,0) AS is_stm,
        COALESCE(CASE
            WHEN ROW_NUMBER() OVER (
                PARTITION BY a.season, i.email
                ORDER BY i.program
            ) = 1
            THEN s.is_stm
            ELSE 0
        END,0) AS is_stm_dedup
    FROM
        initial i
    LEFT JOIN
        acct_rev_info a
        ON i.email = a.email
    LEFT JOIN
        subs s
        ON i.email = s.email
        AND a.season = s.season
    WHERE
        i.email IS NOT NULL
        AND i.email <> ''
"""

df = FLA_Redshift(**rs_creds).query_warehouse(sql_string = q)

In [4]:
df

,email,program_group,program,first_program_season,rev_stm_season,email_status,program_data,gross_rev,gross_rev_dedup,is_stm,is_stm_dedup
0,2many@web.de,LTP,LTP,2020-21,2021-22,email_seen_before,post_program_data,2113.919994,2113.919994,0,0
1,5floridafans@gmail.com,LTP,LTP,2017-18,2021-22,new_email,post_program_data,586.000000,586.000000,0,0
2,84justinhughes@gmail.com,Kids Club,Rookie,2022-23,2021-22,email_seen_before,pre_program_data,64.000000,64.000000,0,0
3,a.davitkov@yahoo.com,Kids Club,All-Star,2024-25,2021-22,email_seen_before,pre_program_data,188.000000,188.000000,0,0
4,a.davitkov@yahoo.com,LTP,LTP,2023-24,2021-22,email_seen_before,pre_program_data,188.000000,0.000000,0,0
...,...,...,...,...,...,...,...,...,...,...,...
19025,zuaja78@hotmail.com,Kids Club,Rookie,2024-25,None,email_seen_before,post_program_data,0.000000,0.000000,0,0
19026,zubman007@yahoo.com,Kids Club,Rookie,2024-25,None,email_seen_before,post_program_data,0.000000,NaN,0,0
19027,zuckerma1190@yahoo.com,Kids Club,All-Star,2024-25,None,email_seen_before,post_program_data,0.000000,NaN,0,0
19028,zulickdavidm@gmail.com,LTP,LTP,2015-16,None,email_seen_before,post_program_data,0.000000,NaN,0,0
